In [24]:
import pandas as pd
import ast


In [25]:

#cambiar como sea relevante la direccion. esto asume esta en el mismo directorio
df = pd.read_csv("./libros.csv")

print(f"Cantidad de libros: {len(df)}")
df.head()

Cantidad de libros: 120


,id,titulo,autores,generos,serie,sinopsis,url_libro,categoria_origen,fecha_extraccion
0,1,El papá de las bellezas,"[""Felipe Trigo""]","[""Realista"", ""Relato""]",NaN,"El papá de las bellezas , una obra maestra de ...",https://ww3.lectulandia.com/book/el-papa-de-la...,Realismo,2026-09-11
1,2,La última cerilla,"[""Marie Vareille""]","[""Novela"", ""Psicológico"", ""Realista""]",NaN,Las cifras no mienten: tres de cada cuatro niñ...,https://ww3.lectulandia.com/book/la-ultima-cer...,Realismo,2026-09-11
2,3,La enjuta,"[""Víctor Català""]","[""Realista"", ""Relato""]",NaN,En estos relatos la autora presenta una radiog...,https://ww3.lectulandia.com/book/la-enjuta/,Realismo,2026-09-11
3,4,En el camping,"[""Soledad Puértolas""]","[""Realista"", ""Relato""]",NaN,El esplendor de Soledad Puértolas en su madure...,https://ww3.lectulandia.com/book/en-el-camping/,Realismo,2026-09-11
4,5,Koljós,"[""Emmanuel Carrère""]","[""Ensayo"", ""Memorias"", ""Novela"", ""Realista""]",NaN,"Un libro total, una construcción narrativa des...",https://ww3.lectulandia.com/book/koljos/,Realismo,2026-09-11


In [26]:
def convertir_a_string(valor):
    if pd.isna(valor):
        return ""
    
    if isinstance(valor, str):
        try:
            valor_parseado = ast.literal_eval(valor)
            
            if isinstance(valor_parseado, list):
                return " ".join(str(x) for x in valor_parseado)
            
            return str(valor_parseado)
        
        except (ValueError, SyntaxError):
            return valor
    
    if isinstance(valor, list):
        return " ".join(str(x) for x in valor)
    
    return str(valor)


In [27]:
columnas = [
    "titulo",
    "autores",
    "generos",
    "serie",
    "sinopsis"
]

for columna in columnas:
    df[columna] = df[columna].apply(convertir_a_string)


CAMBIAR LA VARIABLE DE PURO!!


In [28]:
df["puro"] = df[columnas].agg(" ".join, axis=1)

df["puro"] = (
    df["puro"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


In [29]:
df[["titulo", "autores", "generos", "serie", "sinopsis", "puro"]].head()


,titulo,autores,generos,serie,sinopsis,puro
0,El papá de las bellezas,Felipe Trigo,Realista Relato,,"El papá de las bellezas , una obra maestra de ...",El papá de las bellezas Felipe Trigo Realista ...
1,La última cerilla,Marie Vareille,Novela Psicológico Realista,,Las cifras no mienten: tres de cada cuatro niñ...,La última cerilla Marie Vareille Novela Psicol...
2,La enjuta,Víctor Català,Realista Relato,,En estos relatos la autora presenta una radiog...,La enjuta Víctor Català Realista Relato En est...
3,En el camping,Soledad Puértolas,Realista Relato,,El esplendor de Soledad Puértolas en su madure...,En el camping Soledad Puértolas Realista Relat...
4,Koljós,Emmanuel Carrère,Ensayo Memorias Novela Realista,,"Un libro total, una construcción narrativa des...",Koljós Emmanuel Carrère Ensayo Memorias Novela...


limpieza!!

In [30]:
import re

STOPWORDS_ES = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "como",
    "con", "contra", "cual", "cuando", "de", "del", "desde", "donde",
    "dos", "el", "ella", "ellas", "ellos", "en", "entre", "era", "es",
    "esa", "esas", "ese", "eso", "esos", "esta", "estas", "este", "esto",
    "estos", "fue", "ha", "han", "hasta", "hay", "la", "las", "le", "les",
    "lo", "los", "más", "me", "mi", "mis", "muy", "no", "nos", "o", "para",
    "pero", "por", "que", "qué", "se", "sea", "ser", "si", "sí", "sin",
    "sobre", "son", "su", "sus", "también", "te", "tiene", "tienen", "todo",
    "todos", "tu", "tus", "un", "una", "uno", "unos", "y", "ya"
}

def remover_acentos(texto):
    texto = re.sub(r"[áàäâãå]", "a", texto)
    texto = re.sub(r"[éèëê]", "e", texto)
    texto = re.sub(r"[íìïî]", "i", texto)
    texto = re.sub(r"[óòöôõ]", "o", texto)
    texto = re.sub(r"[úùüû]", "u", texto)
    texto = texto.replace("ñ", "n")
    
    return texto

In [31]:
def limpiar_texto(texto):
    # 1. Lowercase
    texto = texto.lower()

    # 2. Eliminar signos de puntuación
    texto = re.sub(r"[^\w\s]", "", texto)

    # 3. Normalizar espacios
    texto = re.sub(r"\s+", " ", texto).strip()

    # 4. Remover acentos
    texto = remover_acentos(texto)
    
    # 5. Remover stopwords
    palabras = texto.split()
    palabras = [
        palabra for palabra in palabras
        if palabra not in STOPWORDS_ES
    ]

    return " ".join(palabras)


# Aplicar a la columna "puro"
df["puro"] = df["puro"].apply(limpiar_texto)

In [32]:
df["puro"].head()

0    papa bellezas felipe trigo realista relato pap...
1    ultima cerilla marie vareille novela psicologi...
2    enjuta victor catala realista relato relatos a...
3    camping soledad puertolas realista relato espl...
4    koljos emmanuel carrere ensayo memorias novela...
Name: puro, dtype: str

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [34]:

# Crear el vectorizador
vectorizador = TfidfVectorizer()

# Calcular TF-IDF
matriz_tfidf = vectorizador.fit_transform(df["puro"])

# Guardar el vector de cada libro en df["vec"]
df["vec"] = list(matriz_tfidf.toarray())

In [35]:
vector = df["vec"].iloc[0]
terminos = vectorizador.get_feature_names_out()

for termino, valor in zip(terminos, vector):
    if valor > 0:
        print(f"{termino}: {valor:.4f}")

ambiente: 0.1235
audaz: 0.1342
belleza: 0.1101
bellezas: 0.2684
bello: 0.1342
cambio: 0.1012
complejidad: 0.1235
contexto: 0.1160
cotidiana: 0.1160
crea: 0.1342
critica: 0.0830
cuidadoso: 0.1342
deseos: 0.1101
despliega: 0.1160
enfoque: 0.1342
entrelaza: 0.1342
entrelazado: 0.1342
espana: 0.0977
estetica: 0.1342
estilo: 0.0946
evocador: 0.1235
evolucion: 0.1235
examinan: 0.1342
felipe: 0.2025
femenina: 0.1053
grotesco: 0.1342
hacia: 0.0894
humanas: 0.0946
humanos: 0.1101
inicios: 0.1342
invita: 0.1053
lector: 0.0764
libro: 0.0657
maestra: 0.1012
motivaciones: 0.1342
narrativo: 0.1053
obra: 0.0678
papa: 0.2684
papel: 0.1101
personajes: 0.0736
presentando: 0.1342
profundamente: 0.1012
propias: 0.1160
publicado: 0.1101
realidad: 0.0795
realista: 0.0263
refleja: 0.1101
reflexion: 0.1160
relaciones: 0.0946
relato: 0.0597
sensualidad: 0.1160
siglo: 0.0919
social: 0.1837
sociedad: 0.0830
sumerge: 0.1160
tensiones: 0.1160
transformaciones: 0.1342
trigo: 0.3159
vibrante: 0.1235
vida: 0.0430
xx:

In [36]:
vector = df["vec"].iloc[0]
terminos = vectorizador.get_feature_names_out()

top_10 = sorted(
    zip(terminos, vector),
    key=lambda x: x[1],
    reverse=True
)[:10]

for termino, valor in top_10:
    print(f"{termino}: {valor:.4f}")

trigo: 0.3159
bellezas: 0.2684
papa: 0.2684
felipe: 0.2025
social: 0.1837
audaz: 0.1342
bello: 0.1342
crea: 0.1342
cuidadoso: 0.1342
enfoque: 0.1342


GUARDAR EL VECTORIZADO


In [37]:
import numpy as np

# Guardar los vectores en un archivo físico
np.savez_compressed(
    "vectores_libros.npz",
    ids=df["id"].to_numpy(),
    vectores=np.stack(df["vec"].to_numpy())
)



In [38]:
datos = np.load("vectores_libros.npz")

ids = datos["ids"]
vectores = datos["vectores"]

vector = vectores[0]
terminos = vectorizador.get_feature_names_out()

top_10 = sorted(
    zip(terminos, vector),
    key=lambda x: x[1],
    reverse=True
)[:10]

for termino, valor in top_10:
    print(f"{termino}: {valor:.4f}")

trigo: 0.3159
bellezas: 0.2684
papa: 0.2684
felipe: 0.2025
social: 0.1837
audaz: 0.1342
bello: 0.1342
crea: 0.1342
cuidadoso: 0.1342
enfoque: 0.1342


PARTE 